# Reward Model 

In this notebook we build a **minimal but real reward‑modeling setup** that runs **on CPU only**.

We will:

1. ✅ **Train a reward model (RM)** on real human preference data.
2. ✅ **Probe the trained RM** with a few hand‑crafted prompts/responses.

### Quick intuition: RLHF and reward models

- In **reinforcement learning (RL)**, an agent takes actions, gets **rewards**, and updates itself to get higher rewards in the future.
- In **RLHF (Reinforcement Learning from Human Feedback)**, the reward comes from **people**:
  - Humans compare responses (A vs B) for the same prompt.
  - We train a **reward model** that predicts which answer humans prefer.
  - In a *full* RLHF pipeline, we would then run RL (e.g. **PPO**) on the language model so it learns to produce answers that the **reward model scores highly**.

In **this** notebook we **only train and inspect the reward model**. We stop before the PPO/RL step.

### Bradley–Terry view of reward models

Most practical reward models for RLHF are trained with a **pairwise preference loss**. The classic formulation is the **Bradley–Terry (BT) model**:

- Let the reward model produce scores `r_chosen` and `r_rejected` for two answers to the same prompt.
- Interpret their *difference* as a logit:
  
p(chosen preferred) = sigmoid( r_chosen - r_rejected )

Where:
sigmoid(x) = 1 / (1 + exp(-x))

- The training objective is to **maximize this probability**, i.e. minimize the negative log‑likelihood:
  
BT_Loss = - log( sigmoid( r_chosen - r_rejected ) )

Intuition:

- If  r_chosen >> r_rejected   -->  Loss is small  (model agrees with humans)
- If  r_chosen <  r_rejected   -->  Loss is large  (model disagrees)

Most modern implementations (including TRL's `RewardTrainer`) use some variant of this BT‑style loss for training reward models on preference data.



## Setup

Install TRL, Transformers, Datasets, and Accelerate.

> I kept everything **small and CPU-friendly**. This is for understanding the wiring, not for competitive performance.
> So no bitsandbytes / CUDA / MPS.

```bash
!pip install -q "trl" "transformers" "datasets" "accelerate" "torch" "tqdm"
```


In [ ]:
import torch

# 🔒 Force CPU-only for this notebook (even if a GPU is available)
device = torch.device("cpu")
print("Using device:", device)

## 1. Reward Modeling from Human Preference Data

Here we:

- Load a small **decoder-only model** for reward modeling.
- Train it on **`trl-lib/ultrafeedback_binarized`**, which contains human preference data.
- The reward model outputs a **single scalar score** per input: higher = more preferred.

> **Conceptually**: the reward model learns to imitate human preferences.

### I am using a decoder‑only model

For this teaching demo we use a **tiny decoder-only LM (`gpt2`) with a scalar head** as the reward model backbone:

- It is small and **CPU-friendly**.
- Many people have already seen GPT-2, so it keeps the code familiar.
- By adding a 1-dimensional classification head, we turn "next-token LM" into a **scoring network** for whole (prompt, response) pairs.

In *production* RLHF systems, people often prefer **encoder-style models** such as
`OpenAssistant/reward-model-deberta-v3-large-v2` as reward models because:

- DeBERTa-style encoders see the whole sequence **bidirectionally**, which is a natural fit for "judge" tasks.
- They are usually **better pre-trained** for classification and ranking tasks than small causal LMs.
- For the same compute budget, they often give **stronger correlation with human preferences**.

So: for **didactic, CPU-only purposes**, a tiny GPT-2 + scalar head is perfect.
If you care about **reward quality**, models like
`OpenAssistant/reward-model-deberta-v3-large-v2` are typically a **better choice than plain GPT-2**, but they are heavier and less friendly for a quick notebook run.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import RewardConfig, RewardTrainer

# A tiny causal LM backbone for the reward model (CPU-friendly)
RM_MODEL_NAME = "gpt2" # Number of parameters: 124M

tokenizer_rm = AutoTokenizer.from_pretrained(RM_MODEL_NAME)
if tokenizer_rm.pad_token is None:
    tokenizer_rm.pad_token = tokenizer_rm.eos_token

# 🔧 Minimal fix: define a simple chat template so RewardTrainer can format (prompt, chosen/rejected)
tokenizer_rm.chat_template = (
    "{% for message in messages %}"
    "{{ message['role'] }}: {{ message['content'] }}\n"
    "{% endfor %}"
)

# Sequence classification head with 1 scalar output (the reward score)
rm_model = AutoModelForSequenceClassification.from_pretrained(
    RM_MODEL_NAME,
    num_labels=1,
)
rm_model.config.pad_token_id = tokenizer_rm.pad_token_id
rm_model.to(device)

print("Reward model loaded on:", next(rm_model.parameters()).device)

### 1.1 Load preference dataset: `trl-lib/ultrafeedback_binarized`

This dataset has **chosen/rejected** responses and is designed for preference-based training
(DPO, reward modeling, etc.). We only take a **small slice** to keep CPU training reasonable.


In [ ]:
# Small subset for CPU demo
rm_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:128]")
print(rm_dataset[0].keys())
rm_dataset[0]

### 1.2 Train the reward model (very small run)

> We use TRL's `RewardTrainer`, which wraps the standard Transformers Trainer for preference-style reward models.

- No GPUs
- Small batch size
- Only a few hundred examples

This is enough to **show the wiring** from dataset → reward model.


In [ ]:
rm_config = RewardConfig(
    output_dir="./rm-gpt2-ultrafeedback",  # where to save the reward model
    per_device_train_batch_size=2,          # small for CPU
    # We keep other args at their defaults to stay compatible with many TRL versions
)

rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_config,
    train_dataset=rm_dataset,
    eval_dataset=None,
    processing_class=tokenizer_rm,  # tokenizer for preprocessing
)

rm_trainer.train()

# Save reward model and tokenizer
rm_trainer.save_model("./rm-gpt2-ultrafeedback")
tokenizer_rm.save_pretrained("./rm-gpt2-ultrafeedback")
print("Saved reward model to ./rm-gpt2-ultrafeedback")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert the trainer log history to a DataFrame
logs = pd.DataFrame(rm_trainer.state.log_history)

# Filter to only rows that have a loss value
loss_logs = logs[logs["loss"].notnull()]

plt.figure(figsize=(8,4))
plt.plot(loss_logs["loss"], marker="o")
plt.title("Reward Model Training Loss")
plt.xlabel("Logging Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()


In [ ]:
rm_model.to(device)

def rm_score(prompt, response):
    msgs = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    ids = tokenizer_rm.apply_chat_template(
        msgs, tokenize=True, return_tensors="pt"
    )
    # 🔑 ensure tensors are on the same device as the model
    model_device = next(rm_model.parameters()).device
    ids = ids.to(model_device)

    rm_model.eval()
    with torch.no_grad():
        out = rm_model(input_ids=ids)
        return out.logits[0, 0].item()


# Example 1 — relevant
p1 = "Explain why exercise is good for health."
r1 = "Exercise improves heart health, mood, sleep, and reduces disease risk."
print("reward 1:", rm_score(p1, r1))

# Example 2 — totally off-topic
p2 = "Count from 1 to 10."
r2 = "1000."
print(" reward 2:", rm_score(p2, r2))


At this point, we have a **tiny reward model** that takes in text and outputs a scalar score,
trained on real preference data (a small subset of UltraFeedback).

In a **full RLHF pipeline**, the next step would be to plug this reward model into an
RL algorithm such as **PPO** and fine-tune a policy model to **maximize the reward**.

## 3. Summary

In this CPU-only notebook we built a **reward-modeling setup** with tiny models and a real RLHF dataset:

1. ✅ **Reward model training**
   - Backbone: a small decoder-only LM (`gpt2`) with a scalar classification head.
   - Data: `trl-lib/ultrafeedback_binarized` (human preference data with chosen/rejected responses).
   - Objective: a **Bradley–Terry style pairwise loss** that pushes the model to score human-preferred answers higher.
   - Trainer: TRL's `RewardTrainer`.

2. ✅ **Inspecting the trained RM**
   - We wrapped the model in a tiny `rm_score(prompt, response)` helper.
   - We checked that it tends to give **higher scores to good, on-topic answers** than to bad or empty ones.
